# Counts Matrix QC and Normalisation

This notebook demonstrates how to perform basic quality control and pre-processing steps on a counts matrix, in order to make the dataset suitable for downstream applications.

Limitation: This notebook is applicable to droplet single-cell RNA-seq experiments.  

Authors: 
Sergio Forcelloni

## Table of contents  
0. [Background](#background)
1. [Reading data](#reading_data)
2. [Gene annotation](#gene_annotation)
3. [Filtering](#filtering)  
    3.1 [Filtering barcodes](#filtering_barcodes)  
    3.2 [Filtering genes](#filtering_genes)  
    3.3 [Doublet removal](#doublet_removal)  
4. [Normalisation](#normalisation)
5. [Merge datasets](#merging)

## 0. Background

### Data

We are working with scRNAseq data on medulloblastoma cells, investigating the transcriptional profile of these cells extracted from four consensus molecular subgroups: WNT, SHH, Group 3, Group 4.

### Setting up

In [ ]:
# package dependencies and suppress unnecessary warning messages 
from numba.core.errors import NumbaDeprecationWarning, NumbaPendingDeprecationWarning, NumbaWarning
import warnings

warnings.simplefilter('ignore', category=NumbaDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaPendingDeprecationWarning)
warnings.simplefilter('ignore', category=NumbaWarning)

In [ ]:
import os
from glob import glob
import numpy as np
from scipy.stats import median_abs_deviation
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import anndata as ad
import pybiomart as bm
import seaborn as sns

# set plotting theme
sns.set_theme()

<a id="reading_data"></a>
## 1. Reading data

We use the filtered outputs of CellRanger.  We start by reading the data using one of the read functions from `scanpy` ([documented here](https://scanpy.readthedocs.io/en/stable/api/reading.html)).

In [ ]:
# You may specify your specific input and output directories when recreating this analysis on your local machine or HPC.
prefix_inputs = "/archivio_immuno/NGS_master_folder/NGS_data/Riemondy2022_PMID34077540_scRNAseq_Nazio_Medulloblastoma"
prefix_outputs = "./Data/results/preprocessing"

os.makedirs(prefix_outputs, exist_ok=True)

# Load the compressed CSV file
file_path = f"{prefix_inputs}/GSE155446_human_raw_counts.csv.gz"

# Read the file, using the first column as the index (genes) and the first row as the header (cells)
df = pd.read_csv(file_path, index_col=0)

# Convert the dataframe into an AnnData object
adata = sc.AnnData(df.T)  # Transpose the matrix to have cells as rows and genes as columns

In [ ]:
# Set gene names as the variable index in adata.var
# The original DataFrame has genes as rows, so we can use 'adata.var_names'
adata.var['gene_name'] = adata.var_names  # Add a 'gene_name' column to adata.var containing the gene names

# Set 'gene_name' as the index of adata.var
adata.var.set_index('gene_name', inplace=False)

# Verify that the adata.var index contains the gene names
print(adata.var.head())

Write the AnnData on a H5AD file.

In [ ]:
adata.write("GSE155446_human_raw_counts.h5ad")

In [ ]:
adata

In [ ]:
adata.var

In [ ]:
adata.obs

Read the sample information and medulloblastoma subtype:

In [ ]:
# Path to the metadata file
metadata_file = "GSE155446_human_cell_metadata.csv.gz"

# Load the metadata as a DataFrame
metadata = pd.read_csv(metadata_file, index_col=0)  # Use the 'cell' column as the index

# Select only the columns of interest
metadata_subset = metadata[['subgroup', 'geo_sample_id']]

# Make sure the indices match the cells in adata
adata.obs = adata.obs.join(metadata_subset, how="left")

# Check the result
print(adata.obs.head())

For the following steps, we need to apply the filters to each sample separately:

In [ ]:
adata

In [ ]:
# Create a dictionary to separate the data based on geo_sample_id
sample_dict = {}

# Iterate over the unique geo_sample_id values and create an AnnData object for each sample
for geo_id in adata.obs['geo_sample_id'].unique():
    # Select the cells belonging to this geo_sample_id
    cells_in_sample = adata[adata.obs['geo_sample_id'] == geo_id]
    
    # Add the AnnData object to the dictionary using geo_sample_id as the key
    sample_dict["sample_"+geo_id] = cells_in_sample

# Check the result
print(f"Numero di campioni nel dizionario: {len(sample_dict)}")
print(f"Esempio di chiavi nel dizionario: {list(sample_dict.keys())[:5]}")  # Stampa le prime 5 chiavi

<a id="gene_annotation"></a>
## 2. Gene Annotation

In [ ]:
adata.var_names

We can see that our index contains the gene name. 
For each of these identifiers, we would like to know information about common gene names and also the chromosome, so we can identify mitochondrial genes. 

We will use the `pybiomart` package, which accesses the Biomart database containing information about genes for many species. 
You can learn more about the package usage from [its documentation](https://jrderuiter.github.io/pybiomart/usage.html). 

In [ ]:
# connect to the Human genes database (GRCh38.p14)
h38_mart = bm.Dataset(name="hsapiens_gene_ensembl",
                      host="http://www.ensembl.org")

# retrieve gene information
h38_genes = h38_mart.query(attributes=["ensembl_gene_id", "external_gene_name", "chromosome_name"])

# rename the columns
h38_genes = h38_genes.rename(columns={"Gene stable ID": "gene_ids", "Gene name": "gene_name", "Chromosome/scaffold name": "chrom"})

This returns a Pandas DataFrame object: 

In [ ]:
h38_genes

We can now merge this DataFrame with the DataFrame from our AnnData metadata:  

In [ ]:
def annotation(sample_dict_):
    dict_tmp = {}
    
    # Iterate over the keys and values of the dictionary
    for geo_sample_id, sample in sample_dict_.items():
        # 'geo_sample_id' is the key (i.e., the 'geo_sample_id')
        # 'sample' is the value (the AnnData object associated with that sample)
        
        print(f"Processing sample with geo_sample_id: {geo_sample_id}")
        print(f"AnnData shape: {sample.shape}")
        
        # Merge while keeping only the corresponding genes and removing duplicates
        gene_annot = (
            sample.var
            .merge(h38_genes, how="left", on="gene_name")
            .drop_duplicates(subset="gene_name")
        )
        
        # Set the 'gene_name' column as the index
        gene_annot.set_index('gene_name', inplace=True)
        
        # Filter to keep only the variables present in sample.var_names
        # We re-annotate our AnnData, being careful to ensure that the order of the genes
        # is the same as in the original AnnData object: 
        gene_annot = gene_annot.loc[gene_annot.index.intersection(sample.var_names)]
        
        # Assign the annotation to sample.var, ensuring that the dimensions match
        if len(gene_annot) == len(sample.var):
            sample.var = gene_annot
        else:
            print(f"ATTENZIONE: dimensione non corrispondente per {sample}. Skipping.")

        # Ensure variable names are unique to avoid warnings
        sample.var_names_make_unique()

        # Filter to keep only the variables present in sample.var_names
        # We re-annotate our AnnData, being careful to ensure that the order of the genes
        # is the same as in the original AnnData object: 
        vars_to_keep = sample.var["chrom"].isin([str(i) for i in range(1, 23)] + ["X", "Y", "MT"])
        sample = sample[:, vars_to_keep].copy()
        
        # Add mitochondrial gene annotation
        # Finally, we create a variable in our gene metadata to indicate whether
        # a gene is mitochondrial or not. We will use this later on during quality control.
        sample.var["mt"] = sample.var["chrom"] == "MT"
        
        dict_tmp[geo_sample_id]=sample

    return dict_tmp

sample_dict_1 = annotation(sample_dict)

In [ ]:
sample_dict_1

<a id="filtering"></a>
## 3. Filtering

<a id="filtering_genes"></a>
### 3.1 Filtering Barcodes

We start by doing some exploratory analysis of our raw count data, namely in terms of:  

- number of total counts per barcode  
- number of detected genes per barcode  
- fraction of counts in mitochondrial genes  

In [ ]:
def filtering(sample_dict_):
    # Initialize an empty list to store processed samples
    dict_tmp = {}
    
    for geo_sample_id, sample in sample_dict_.items():
        # Calculate various quality control metrics for each sample (e.g., mitochondrial genes)
        sc.pp.calculate_qc_metrics(
            sample, 
            qc_vars=["mt"], # Mitochondrial genes are included in the QC metrics
            inplace=True, # Modify the sample object in place
            percent_top=[20], # Use top 20% of genes for QC calculation
            log1p=True # Apply log1p transformation to the data
        )
        # Since sample.obs is a regular DataFrame, we can use standard plotting libraries to visualise these statistics.
        # For example, using the popular Seaborn library:
        # Plot distributions for total counts and mitochondrial percentage
        sns.displot(sample.obs, x="total_counts", bins=100)
        sns.displot(sample.obs, x="pct_counts_mt", bins=100)
        # Create a scatter plot to show the relationship between total counts, number of genes by counts, and mitochondrial percentage
        sns.scatterplot(sample.obs, x="total_counts", y="n_genes_by_counts", hue="pct_counts_mt")

        # Ensure that gene names are unique to avoid conflicts in later plots
        sample.var_names_make_unique()

        # Alternatively, we can use scanpy's own plotting functions (histogram is not available, but we can do violin plots instead):
        # Generate violin plots for total counts and mitochondrial percentage
        sc.pl.violin(sample, "total_counts", xlabel='total counts', ylabel='occurrencies', save=f"_{geo_sample_id}_total_counts.png")
        sc.pl.violin(sample, "pct_counts_mt", xlabel='pct counts mt', ylabel='occurrencies', save=f"_{geo_sample_id}_pct_counts_mt.png")
        # Create a scatter plot to visualize the relationship between total counts and number of genes, colored by mitochondrial percentage
        sc.pl.scatter(sample, "total_counts", "n_genes_by_counts", color="pct_counts_mt", save=f"_{geo_sample_id}_total_counts.png")

        # We can even do several violin plots at once.
        # Create a multi-panel violin plot for total counts, number of genes by counts, and mitochondrial percentage
        sc.pl.violin(
          sample,
          ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
          multi_panel=True, 
          save=f"_{geo_sample_id}_all_together.png"
        )

        # Define a function to identify outliers based on a specified metric and number of median absolute deviations (MADs)
        def is_outlier(adata, metric: str, nmads: int):
          M = adata.obs[metric]
          # Define the outlier condition using median and MAD
          outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
              np.median(M) + nmads * median_abs_deviation(M) < M
          )
          return outlier

        # The function returns True or False depending on whether the value exceeds the specified value of median absolute deviation. 
        # For example:
        # Identify counts outliers and add a new column to the sample metadata
        sample.obs["counts_outlier"] = is_outlier(sample, "log1p_total_counts", 5)
        # Visualize the distribution of log-transformed total counts, showing outliers in a stacked format
        sns.displot(sample.obs, x="log1p_total_counts", hue="counts_outlier", multiple="stack")
        
        # Note that we used the log-transformed counts, as its distribution is less skewed and therefore more suitable for the MAD-based 
        # filtering we are doing. We can repeat this for number of detected genes (also log-transformed) and the percentage of counts in 
        # the top 20 genes:
        
        # Identify genes outliers and add a new column to the sample metadata
        sample.obs["genes_outlier"] = is_outlier(sample, "log1p_n_genes_by_counts", 5)
        sns.displot(sample.obs, x="log1p_n_genes_by_counts", hue="genes_outlier", multiple="stack")
        
        # Identify top genes outliers and add a new column to the sample metadata
        sample.obs["topgenes_outlier"] = is_outlier(sample, "pct_counts_in_top_20_genes", 5)
        sns.displot(sample.obs, x="pct_counts_in_top_20_genes", hue="topgenes_outlier", multiple="stack")

        # We also check for outliers with regards to percentage of mitochondrial counts, where we use more strict filters.
        # Identify mitochondrial outliers based on the percentage of mitochondrial counts and flag samples with more than 8% mitochondrial reads
        sample.obs["mito_outlier"] = is_outlier(sample, "pct_counts_mt", 5)
        # Display the count of outlier samples
        sns.displot(sample.obs, x="pct_counts_mt", hue="mito_outlier", multiple="stack")
        
        # Finally, we create a variable which is the union of these conditions, i.e. if the barcode is determined an outlier of any of our filters, 
        # then we consider it to be an outlier:
        sample.obs["outlier"] = sample.obs["genes_outlier"] | sample.obs["counts_outlier"] | sample.obs["topgenes_outlier"] | sample.obs["mito_outlier"]
        sample.obs["outlier"].value_counts()

        # We can visualise our scatterplot of counts vs detected genes to see which barcodes will be removed:
        # Scatter plot for total counts vs. number of genes by counts, colored by the outlier status
        sns.scatterplot(sample.obs, 
                        x = "total_counts", 
                        y = "n_genes_by_counts",
                        hue = "outlier")
        # Rimuove le cellule outlier
        sample = sample[~sample.obs["outlier"]]  
        
        # Append the processed sample to the list            
        dict_tmp[geo_sample_id]=sample
        
    # Return the processed list of samples
    return dict_tmp
    
# Call the filtering function and store the result  
sample_dict_2 = filtering(sample_dict_1)

In [ ]:
sample_dict_2

<a id="filtering_genes"></a>
### 3.2 Filtering Genes

We will remove undetected genes, i.e. those with zero total counts. We can use the `sc.pp.filter_genes()` function to do this:  

In [ ]:
def removing_genes_and_cells(sample_dict_):

    dict_tmp = {}
    for geo_sample_id, sample in sample_dict_.items():
        # count of genes with zero counts
        sample.var["total_counts"].eq(0).value_counts()
        # count of genes with zero counts
        sc.pp.filter_genes(sample, min_counts=0)
        # Remove cells with zero total counts to avoid unexpected behaviour in the downstream analyses
        sample = sample[sample.obs["total_counts"] > 0, :].copy()
        # Append the processed sample to the list        
        dict_tmp[geo_sample_id]=sample
    # Return the processed list of samples
    return dict_tmp

sample_dict_3 = removing_genes_and_cells(sample_dict_2)

In [ ]:
sample_dict_3

<a id="doublet_removal"></a>
### 3.3 Doublet removal

In this special barcode filtering step, we remove droplet barcodes that may contain more than one cell or nucleus. This is an important step because, unremoved, these droplets known as "doublets" can be misclassified and thus confound downstream analysis.  

Here we run a doublet-detection algorithm that is available in scanpy, called Scrublet ([Wolock et al., 2019](https://doi.org/10.1016/j.cels.2018.11.005)).

Notice that running Scrublet has added `predicted_doublet` and `doublet_score` to `adata.obs`. We can check how many have been predicted as doublets by Scrublet.

Here, we filter out the doublets before moving forward.  Below, the code retains all the barcodes where `predicted_doublet` is False.

In [ ]:
def removing_doublets(sample_dict_):

    dict_tmp = {}
    for geo_sample_id, sample in sample_dict_.items():
        # Make sure we are working on a copy of the AnnData object to avoid modifying a view
        sample = sample.copy()
        # print the how our cell barcode metadata container, .obs, looks like at the moment. 
        # It has outputs saed from the previous filtering steps.
        print (sample.obs.keys())
        # Here we run a doublet-detection algorithm that is available in scanpy, called Scrublet (Wolock et al., 2019).
        sc.pp.scrublet(sample)
        # Notice that running Scrublet has added predicted_doublet and doublet_score to ETV6_RUNX1_1.obs. 
        # We can check how many have been predicted as doublets by Scrublet.
        print (sample.obs.keys())
        print (sample.obs["predicted_doublet"].value_counts())
        # Ensure 'predicted_doublet' is boolean and fill NaN with False
        sample.obs["predicted_doublet"] = sample.obs["predicted_doublet"].fillna(False).astype(bool)
        # One can use the above results by filtering out the cells called as doublets before moving on to the next step. 
        # Another option is to wait until after clustering, and filter out clusters with high doublet scores (reference). 
        # There is also no reason to be limited by one doublet detection method--one can use multiple methods, and filter 
        # out the barcodes called as doublets by the different methods. Here, we filter out the doublets before moving forward. 
        # Below, the code retains all the barcodes where predicted_doublet is False.
        sample = sample[~sample.obs["predicted_doublet"], :].copy()
        # Append the processed sample to the list        
        dict_tmp[geo_sample_id]=sample
    # Return the processed list of samples
    return dict_tmp
    
sample_dict_4 = removing_doublets(sample_dict_3)

In [ ]:
sample_dict_4

<a id="normalisation"></a>
## 4. Normalisation

For normalization, raw count data were first preserved in a separate counts layer, while a copy of the raw expression matrix was stored in the logcounts layer for downstream processing. The expression values were then normalized on a per-cell basis using sc.pp.normalize_total, with a target library size equal to the median of total counts for observations (cells) before normalization, to account for differences in sequencing depth across cells. Subsequently, a log1p transformation was applied using sc.pp.log1p, resulting in log-normalized expression values that were used for downstream analyses.

In [ ]:
def normalization(sample_dict_):
    # Initialize an empty dictionary to store the processed samples
    dict_tmp = {}

    # Iterate over each sample in the provided list of samples
    
    for geo_sample_id, sample in sample_dict_.items():

        # Store the raw counts (original data) as a backup in a new layer
        sample.layers["counts"] = sample.X.copy()

        # Create a new layer for storing log-normalized counts
        sample.layers["logcounts"] = sample.X.copy()

        # Normalize the total counts in the dataset to the median of total counts for observations (cells) before normalization.
        sc.pp.normalize_total(sample, layer="logcounts")

        # Apply log1p transformation to the normalized counts (log(x + 1) transformation)
        sc.pp.log1p(sample, layer="logcounts")

        # Append the processed sample to the list        
        dict_tmp[geo_sample_id]=sample
    # Return the processed list of samples
    return dict_tmp

# Call the normalization function and store the result
sample_dict_5 = normalization(sample_dict_4)

<a id="Merge datasets"></a>
## 5. Merge datasets

In [ ]:
# Concatenate all the samples from the 'adata' dictionary
# 'sc.concat()' combines AnnData objects, and 'join="outer"' ensures all genes from all samples are kept
adata_merged = ad.concat(list(sample_dict_5.values()), 
                         join="outer",  # Non antiene solo i geni comuni a tutti i campioni
                         merge="same",  # Mantiene lo stesso formato di `obs` e `var`
                         label="sample",# 'label="sample"' adds a column to the 'obs' DataFrame to indicate the sample each cell belongs to
                         index_unique="-" # 'index_unique="-" ensures unique indexing in the concatenated object
                        )

# Check the results
print(adata_merged)
print(f"Numero totale di cellule: {adata_merged.n_obs}")
print(f"Numero totale di geni: {adata_merged.n_vars}")

# Print the 'sample' column from the 'obs' DataFrame, which contains the sample information
print(f"Campioni disponibili nell'analisi: {adata_merged.obs["geo_sample_id"].unique()}")

# Write the concatenated AnnData object to an HDF5 file for future use
# This will save all the processed data into a file at the specified path
adata_merged.write(f"./Data/results/preprocessing/normalized_and_preprocessed.h5ad")